In [1]:
# ==========================================================
# Notebook 08: Bias Mitigation
# ==========================================================

import re
import pandas as pd

In [3]:
sections_df = pd.read_csv("processed/resume_sections.csv")

sections_df.head()

,file_name,skills,experience,education,certifications
0,10554236.pdf,skills accounting; general accounting; account...,experience company name july 2011 to november ...,education northern maine community college 199...,certifications certified defense financial man...
1,10674770.pdf,skills. highlights dba quick books mas - sage ...,experience staff accountant january 2014 to oc...,"education bachelor of science : accounting , m...",NaN
2,11163645.pdf,skills and attributes. attributes self-motivat...,experience accountant january 2011 to november...,education computer applications specialist cer...,NaN
3,11759079.pdf,skills by drafting over forty memorandums that...,experience company name june 2011 to current s...,"education emory university, goizueta business ...",certifications and awards fulton county casa b...
4,12065211.pdf,skills aderant/cms financial reporting excel u...,experience in full life cycle of general ledge...,education bachelor of business administration ...,"certifications certified public accountant, ne..."


In [4]:
sample_resume = """
John Smith

Email: johnsmith@gmail.com
Phone: +91-9876543210

Location: Bangalore, India

Education:
Stanford University
B.Tech Computer Science

Skills:
Python, SQL, Machine Learning, NLP

Experience:
5 Years Data Scientist
"""

In [5]:
def remove_email(text):

    return re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", "[EMAIL]", text)

In [6]:
clean_text = remove_email(sample_resume)

print(clean_text)


John Smith

Email: [EMAIL]
Phone: +91-9876543210

Location: Bangalore, India

Education:
Stanford University
B.Tech Computer Science

Skills:
Python, SQL, Machine Learning, NLP

Experience:
5 Years Data Scientist



In [7]:
def remove_phone(text):

    return re.sub(r"(\+?\d[\d\s-]{8,}\d)", "[PHONE]", text)

In [8]:
clean_text = remove_phone(clean_text)

print(clean_text)


John Smith

Email: [EMAIL]
Phone: [PHONE]

Location: Bangalore, India

Education:
Stanford University
B.Tech Computer Science

Skills:
Python, SQL, Machine Learning, NLP

Experience:
5 Years Data Scientist



In [9]:
def remove_first_line_name(text):

    lines = text.split("\n")

    if len(lines) > 0:

        lines[0] = "[CANDIDATE_NAME]"

    return "\n".join(lines)

In [10]:
clean_text = remove_first_line_name(clean_text)

print(clean_text)

[CANDIDATE_NAME]
John Smith

Email: [EMAIL]
Phone: [PHONE]

Location: Bangalore, India

Education:
Stanford University
B.Tech Computer Science

Skills:
Python, SQL, Machine Learning, NLP

Experience:
5 Years Data Scientist



In [11]:
COMMON_LOCATIONS = [
    "bangalore",
    "bengaluru",
    "mumbai",
    "hyderabad",
    "delhi",
    "chennai",
    "india",
    "usa",
]

In [12]:
def remove_locations(text, locations):

    for location in locations:

        pattern = rf"\b{location}\b"

        text = re.sub(pattern, "[LOCATION]", text, flags=re.IGNORECASE)

    return text

In [13]:
clean_text = remove_locations(clean_text, COMMON_LOCATIONS)

print(clean_text)

[CANDIDATE_NAME]
John Smith

Email: [EMAIL]
Phone: [PHONE]

Location: [LOCATION], [LOCATION]

Education:
Stanford University
B.Tech Computer Science

Skills:
Python, SQL, Machine Learning, NLP

Experience:
5 Years Data Scientist



In [14]:
UNIVERSITIES = ["stanford university", "mit", "harvard university", "iit", "iisc"]

In [15]:
def remove_universities(text, universities):

    for uni in universities:

        text = re.sub(
            rf"\b{re.escape(uni)}\b", "[UNIVERSITY]", text, flags=re.IGNORECASE
        )

    return text

In [16]:
clean_text = remove_universities(clean_text, UNIVERSITIES)

print(clean_text)

[CANDIDATE_NAME]
John Smith

Email: [EMAIL]
Phone: [PHONE]

Location: [LOCATION], [LOCATION]

Education:
[UNIVERSITY]
B.Tech Computer Science

Skills:
Python, SQL, Machine Learning, NLP

Experience:
5 Years Data Scientist



In [17]:
def anonymize_resume(text):

    text = remove_email(text)

    text = remove_phone(text)

    text = remove_first_line_name(text)

    text = remove_locations(text, COMMON_LOCATIONS)

    text = remove_universities(text, UNIVERSITIES)

    return text

In [18]:
anonymized_resume = anonymize_resume(sample_resume)

print(anonymized_resume)

[CANDIDATE_NAME]
John Smith

Email: [EMAIL]
Phone: [PHONE]

Location: [LOCATION], [LOCATION]

Education:
[UNIVERSITY]
B.Tech Computer Science

Skills:
Python, SQL, Machine Learning, NLP

Experience:
5 Years Data Scientist



In [19]:
sections_df["anonymized_resume"] = (
    sections_df["skills"].fillna("")
    + " "
    + sections_df["experience"].fillna("")
    + " "
    + sections_df["education"].fillna("")
)

In [20]:
sections_df["anonymized_resume"] = sections_df["anonymized_resume"].apply(
    anonymize_resume
)

sections_df.head()

,file_name,skills,experience,education,certifications,anonymized_resume
0,10554236.pdf,skills accounting; general accounting; account...,experience company name july 2011 to november ...,education northern maine community college 199...,certifications certified defense financial man...,[CANDIDATE_NAME]
1,10674770.pdf,skills. highlights dba quick books mas - sage ...,experience staff accountant january 2014 to oc...,"education bachelor of science : accounting , m...",NaN,[CANDIDATE_NAME]
2,11163645.pdf,skills and attributes. attributes self-motivat...,experience accountant january 2011 to november...,education computer applications specialist cer...,NaN,[CANDIDATE_NAME]
3,11759079.pdf,skills by drafting over forty memorandums that...,experience company name june 2011 to current s...,"education emory university, goizueta business ...",certifications and awards fulton county casa b...,[CANDIDATE_NAME]
4,12065211.pdf,skills aderant/cms financial reporting excel u...,experience in full life cycle of general ledge...,education bachelor of business administration ...,"certifications certified public accountant, ne...",[CANDIDATE_NAME]


In [22]:
sections_df.to_csv("processed/" "fair_resume_dataset.csv", index=False)

print("Bias-mitigated dataset saved.")

Bias-mitigated dataset saved.
